source API URL : "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

JSON Target File Path : "abfss://bronze@datalakestorageaccountname.dfs.core.windows.net/geo-location/"

In [0]:
geo_location_source_API_URL = "https://geocoding-api.open-meteo.com/v1/search?name=kovilpatti&count=10&language=en&format=json"

geo_location_source_API_base_URL = "https://geocoding-api.open-meteo.com/v1/search?name="
geo_location_source_API_URL_options = "&count=10&language=en&format=json"

geo_location_sink_layer_name = 'adbbronze'
geo_location_sink_storage_account_name = 'adbstorageu'
geo_location_sink_folder_name = 'geo-location'

geo_location_sink_folder_path = f"abfss://{geo_location_sink_layer_name}@{geo_location_sink_storage_account_name}.dfs.core.windows.net/{geo_location_sink_folder_name}"

## In the abover case we are defaulting to the location of Kovilpatti but there are more than 1000 locations in the dataset, we need to modularize the code

In [0]:
import requests
import json
import pandas as pd

In [0]:
geo_location_API_response = requests.get(geo_location_source_API_URL).json()
geo_location_pandas_DF = pd.DataFrame(geo_location_API_response)
geo_location_spark_DF = spark.createDataFrame(geo_location_pandas_DF)

In [0]:
daily_pricing_market_names_DF = spark.sql("SELECT MARKET_NAME from adb_rtp.gold.reporting_dim_market_gold")

In [0]:
from pyspark.sql.types import *
market_names = [daily_pricing_market_names["MARKET_NAME"] for daily_pricing_market_names in daily_pricing_market_names_DF.collect()]

##need to convert spark dataframe variable to python array

geo_location_API_response_list = []
for market_name in market_names:
  
  geo_location_source_API_URL = f"{geo_location_source_API_base_URL}{market_name}{geo_location_source_API_URL_options}"
  geo_location_API_response = requests.get(geo_location_source_API_URL).json()
  
  
  if isinstance(geo_location_API_response, dict):
      geo_location_API_response_list.append(geo_location_API_response)
  else:
      print(f"Error: {geo_location_API_response}")    


geo_location_spark_RDD =sc.parallelize(geo_location_API_response_list)

geo_location_spark_DF = spark.read.json(geo_location_spark_RDD)

(geo_location_spark_DF
.filter("results.admin1 IS NOT NULL")
.write
.mode("overwrite")
.json(geo_location_sink_folder_path))

#geo_location_pandas_DF = pd.DataFrame(geo_location_API_response_list)
#geo_location_spark_DF = spark.createDataFrame(geo_location_pandas_DF)
#geo_location_spark_DF.write.mode("overwrite").json(geoLocationSinkFolderPath)     
## The api response here has complex data which might not be handled well by pandas dataframe, hence we are using Spark RDD

In [0]:
from pyspark.sql.functions import col, array_contains
geo_location_bronze_DF = (spark
                       .read
                                             .json(geo_location_sink_folder_path))
                       

display(geo_location_bronze_DF)